# Get Bioavailable Iron from USDA

In [252]:
# Parameters
SAVE_DFS = True

In [253]:
import ast
import os
import pandas as pd
import requests

from dotenv import load_dotenv
from IPython.display import Image
from pathlib import Path
from rapidfuzz import process, fuzz
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [254]:
# Load splits from CSVs
df_train = pd.read_csv('../data/train/df_train.csv')
df_val = pd.read_csv('../data/val/df_val.csv')
df_test = pd.read_csv('../data/test/df_test.csv')

df_train.head(2)

,image_url,camera_or_phone_prob,food_prob,dish_name,food_type,ingredients,portion_size,nutritional_profile,cooking_method,sub_dt,image_name
0,https://file.b18a.io/7832973280900104501_54585...,0.8,0.90,oysters,homemade food,['oysters'],{'oysters': '500g'},"{'fat_g': 5.0, 'protein_g': 20.0, 'calories_kc...",raw,20250710,7832973280900104501_545859_.jpeg
1,https://file.b18a.io/7835136777400102715_70587...,0.7,0.95,grilled steak,restaurant food,"['steak', 'broccoli', 'potato', 'tomato', 'sau...","{'steak': '250g', 'broccoli': '50g', 'potato':...","{'fat_g': 30.0, 'protein_g': 50.0, 'calories_k...",grilling,20250702,7835136777400102715_705873_.jpeg


In [255]:
# Combine dfs
df_all = pd.concat([df_train, df_val, df_test])
df_all = df_all.reset_index()

df_all.shape

(5000, 12)

In [256]:
df_all['dish_name']

0                              oysters
1                        grilled steak
2              sweet and sour potatoes
3                              hot pot
4                   stir-fried noodles
                     ...              
4995                       noodle soup
4996              braised chicken feet
4997                           hot pot
4998    vegetable and chicken sandwich
4999                mixed asian dishes
Name: dish_name, Length: 5000, dtype: object

In [257]:
df_all['portion_size'][0]

"{'oysters': '500g'}"

In [258]:
df_all['portion_size'] = df_all['portion_size'].apply(ast.literal_eval)

df_all['portion_size'][0]

{'oysters': '500g'}

In [259]:
# Get all ingredients
ingredients = set()
for row in df_all.itertuples():
    keys = row.portion_size.keys()
    ingredients.update(row.portion_size.keys())
print(f'{len(ingredients)=}')
ingredients

len(ingredients)=798


{'abalone',
 'almonds',
 'anchovies',
 'apple',
 'apple chips',
 'apples',
 'apricot',
 'apricots',
 'asparagus',
 'assorted desserts',
 'avocado',
 'baby corn',
 'bacon',
 'bagel',
 'baked beans',
 'baked goods',
 'baked potato',
 'baklava',
 'bamboo shoots',
 'banana',
 'bananas',
 'bao dough',
 'bar',
 'base',
 'basil',
 'batter',
 'bbq ribs',
 'bean sprouts',
 'beans',
 'beef',
 'beef liver',
 'beef patty',
 'beef ribs',
 'beef snack',
 'beef stew',
 'beef tripe',
 'beer',
 'beet',
 'beetroot',
 'beets',
 'bell pepper',
 'bell peppers',
 'berries',
 'beverage',
 'beverages',
 'bird',
 'birds',
 'biscuit',
 'biscuits',
 'bitter melon',
 'black beans',
 'black fungus',
 'black jelly',
 'black olive tapenade',
 'black pudding',
 'black rice',
 'black sapote',
 'blackberries',
 'blueberries',
 'boiled egg',
 'boiled eggs',
 'bok choy',
 'borscht',
 'bounty bar',
 'bread',
 'bread roll',
 'bread rolls',
 'breading',
 'brisket',
 'broad beans',
 'broccoli',
 'broth',
 'buckwheat',
 'bulg

In [260]:
nutrient_names = ['Iron, Fe', 'Calcium, Ca', 'Vitamin C, total ascorbic acid']

ingredients_df = {'ingredients': list(ingredients)}
ingredients_df = pd.DataFrame(ingredients_df)
for name in nutrient_names:
    ingredients_df[name] = ''
ingredients_df

,ingredients,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid"
0,dough sticks,,,
1,fried potatoes,,,
2,gravy,,,
3,crisps,,,
4,black rice,,,
...,...,...,...,...
793,scallops,,,
794,shrimp porridge,,,
795,buckwheat,,,
796,gelato,,,


In [261]:
csv_file = '../data/ingredients.csv'

if SAVE_DFS:
    # Save to CSV
    ingredients_df.to_csv(csv_file, index=False)

# Load data from previously saved CSV
ingredients_df = pd.read_csv(csv_file)
ingredients_df

,ingredients,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid"
0,dough sticks,NaN,NaN,NaN
1,fried potatoes,NaN,NaN,NaN
2,gravy,NaN,NaN,NaN
3,crisps,NaN,NaN,NaN
4,black rice,NaN,NaN,NaN
...,...,...,...,...
793,scallops,NaN,NaN,NaN
794,shrimp porridge,NaN,NaN,NaN
795,buckwheat,NaN,NaN,NaN
796,gelato,NaN,NaN,NaN


## USDA FoodData Central API

In [262]:
# Load and get api key from .env
load_dotenv()
api_key = os.getenv("API_KEY")

FOODS_SEARCH_URL = 'https://api.nal.usda.gov/fdc/v1/foods/search'

In [263]:
# Search for food
ingredient = 'porridge'
food_query = ingredient
r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}&dataType=Foundation')
data = r.json()
data_df = pd.DataFrame.from_dict(data['foods'])
if len(data_df) == 0:  # Data not found
    r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}')
    data = r.json()
    data_df = pd.DataFrame.from_dict(data['foods'])
data_df

,fdcId,description,dataType,publishedDate,tags,allHighlightFields,score,microbes,foodNutrients,finalFoodInputFoods,...,packageWeight,servingSizeUnit,servingSize,householdServingFullText,shortDescription,tradeChannels,commonNames,additionalDescriptions,foodCode,foodCategoryId
0,2687788,Potential of moringa leaf and baobab fruit foo...,Experimental,2024-04-18,"[Processing, Post-Harvest, USDA Research]",,303.968050,[],[],[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2417422,"MULTIGRAIN PORRIDGE SLICED BREAD, TARTINE MULT...",Branded,2022-12-22,NaN,,230.002460,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,2 oz/500 g,g,50.0,1 slice,,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
2,2089884,ARTIFICIAL CHICKEN PORRIDGE,Branded,2021-10-28,NaN,<b>Ingredients</b>: <em>PORRIDGE</em> (52%): R...,202.842590,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,4.2 oz/120 g,g,120.0,1 BOWL,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
3,2168938,ORGANIC PORRIDGE OATS,Branded,2021-10-28,NaN,,202.842590,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,35 oz/2.19 LB/992 g,g,40.0,1/2 cup,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
4,1002697,PUMPKIN PORRIDGE WITH HONEY,Branded,2020-06-26,NaN,,202.842590,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,NaN,g,285.0,10.05 ONZ,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
5,552097,RICE PORRIDGE WITH ABALONE,Branded,2019-04-01,NaN,,202.842590,[],"[{'nutrientId': 1004, 'nutrientName': 'Total l...",[],...,NaN,g,285.0,10.05 ONZ,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
6,552435,RICE PORRIDGE WITH TUNA,Branded,2019-04-01,NaN,,202.842590,[],"[{'nutrientId': 1087, 'nutrientName': 'Calcium...",[],...,NaN,g,285.0,10.05 ONZ,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
7,560546,RICE PORRIDGE WITH VEGETABLE,Branded,2019-04-01,NaN,,202.842590,[],"[{'nutrientId': 1004, 'nutrientName': 'Total l...",[],...,NaN,g,285.0,10.05 ONZ,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
8,2089623,SUPER SMOOTH PORRIDGE,Branded,2021-10-28,NaN,,202.842590,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,NaN,g,100.0,100 GRM,NaN,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN
9,2432400,"TEFF PORRIDGE, IVORY",Branded,2022-12-22,NaN,,202.842590,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",[],...,5 oz/142 g,g,50.0,0.25 cup,,[NO_TRADE_CHANNEL],NaN,NaN,NaN,NaN


In [264]:
# Fuzzy string matching
food_names = data_df['description']
best_match = process.extractOne(ingredient, food_names, scorer=fuzz.ratio)

print(f"Ingredient: {ingredient}")
print(f"Best match: {best_match[0]} with a score of {best_match[1]}")
data_df = data_df.loc[data_df['description']==best_match[0]].head(1).reset_index()
data_df

Ingredient: porridge
Best match: Congee with a score of 42.85714285714286


,index,fdcId,description,dataType,publishedDate,tags,allHighlightFields,score,microbes,foodNutrients,...,packageWeight,servingSizeUnit,servingSize,householdServingFullText,shortDescription,tradeChannels,commonNames,additionalDescriptions,foodCode,foodCategoryId
0,27,2708418,Congee,Survey (FNDDS),2024-10-31,NaN,<b>Includes</b>: rice <em>porridge</em> or gruel,26.327248,[],"[{'nutrientId': 1003, 'nutrientName': 'Protein...",...,NaN,NaN,NaN,NaN,NaN,NaN,,rice porridge or gruel,56205101.0,3304384.0


In [265]:
# Get food nutrients
nutrients = pd.DataFrame(data_df.iloc[0]['foodNutrients'])
nutrients = nutrients.loc[nutrients['nutrientName'].isin(nutrient_names)][['nutrientName', 'value']]
nutrients

,nutrientName,value
10,"Calcium, Ca",5.00
11,"Iron, Fe",0.36
28,"Vitamin C, total ascorbic acid",0.00


In [266]:
def get_nutrients(data_df, ingredients_df, nutrient_names):
    if len(data_df) > 0:  # Food found

        # Fuzzy string matching
        food_names = data_df['description']
        best_match = process.extractOne(ingredient, food_names, scorer=fuzz.ratio)
        data_df = data_df.loc[data_df['description']==best_match[0]].head(1).reset_index()
        nutrients = data_df.iloc[0]
        if len(nutrients) > 0:  # Nutrients found
            food_id = nutrients['fdcId']
            food_name = nutrients['description']  # Skip if foodNutrients missing
            nutrients = pd.DataFrame(nutrients['foodNutrients'])
            if len(nutrients) > 0:
                nutrients = nutrients.loc[nutrients['nutrientName'].isin(nutrient_names)][['nutrientName', 'value']]
                for row in nutrients.itertuples():
                    ingredients_df.loc[ingredients_df['ingredients']==ingredient, row.nutrientName] = row.value
                ingredients_df.loc[ingredients_df['ingredients']==ingredient, 'found'] = True
                ingredients_df.loc[ingredients_df['ingredients']==ingredient, 'food_id'] = food_id
                ingredients_df.loc[ingredients_df['ingredients']==ingredient, 'food_name'] = food_name
        
        # nutrients = pd.DataFrame()
        # i = 0
        # while len(nutrients) == 0 and i < len(data_df):  # Look for entry with nutrients
        #     # nutrients = pd.DataFrame(data_df.iloc[i]['foodNutrients'])
        #     nutrients = data_df.iloc[i]
        #     i += 1
        #     if len(nutrients) > 0:  # Nutrients found
        #         food_id = nutrients['fdcId']
        #         food_name = nutrients['description']  # Skip if foodNutrients missing
        #         nutrients = pd.DataFrame(nutrients['foodNutrients'])
        #         if len(nutrients) == 0:
        #             continue
        #         nutrients = nutrients.loc[nutrients['nutrientName'].isin(nutrient_names)][['nutrientName', 'value']]
        #         if len(nutrients) == 0:  # Skip if no desired nutrients
        #             continue
        #         for row in nutrients.itertuples():
        #             ingredients_df.loc[ingredients_df['ingredients']==ingredient, row.nutrientName] = row.value
        #         ingredients_df.loc[ingredients_df['ingredients']==ingredient, 'found'] = True
        #         ingredients_df.loc[ingredients_df['ingredients']==ingredient, 'food_id'] = food_id
        #         ingredients_df.loc[ingredients_df['ingredients']==ingredient, 'food_name'] = food_name
        #         break

    return ingredients_df

In [267]:
# Init food info
ingredients_df['found'] = False
ingredients_df['food_id'] = ''
ingredients_df['food_name'] = ''

In [ ]:
# Fill CSV with nutrient data for ingredients
csv_file = '../data/ingredients_auto.csv'

for ingredient in tqdm(ingredients):
    # Search for food
    food_query = ingredient
    r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}&dataType=Foundation')
    data = r.json()
    data_df = pd.DataFrame.from_dict(data['foods'])
    # Get food nutrients
    ingredients_df = get_nutrients(data_df, ingredients_df, nutrient_names)

    # Check if food not found earlier, broaden food search
    if ingredients_df.loc[ingredients_df['ingredients']==ingredient]['found'].item() == False:
        r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}')
        data = r.json()
        data_df = pd.DataFrame.from_dict(data['foods'])
        # Get food nutrients
        ingredients_df = get_nutrients(data_df, ingredients_df, nutrient_names)

    if SAVE_DFS:
        # Save to CSV
        ingredients_df.to_csv(csv_file, index=False)

# Load data from previously saved CSV
ingredients_df = pd.read_csv(csv_file)
ingredients_df

  1%|          | 9/798 [00:05<06:54,  1.90it/s]

In [ ]:
ingredients_df.loc[ingredients_df['found']==False]

,ingredients,"Iron, Fe","Calcium, Ca","Vitamin C, total ascorbic acid",found,food_id,food_name
26,condiment2,NaN,NaN,NaN,False,NaN,NaN
76,kozinaki,NaN,NaN,NaN,False,NaN,NaN
90,drink3,NaN,NaN,NaN,False,NaN,NaN
93,fruit2,NaN,NaN,NaN,False,NaN,NaN
130,drink1,NaN,NaN,NaN,False,NaN,NaN
137,drink2,NaN,NaN,NaN,False,NaN,NaN
215,drink4,NaN,NaN,NaN,False,NaN,NaN
233,snack1,NaN,NaN,NaN,False,NaN,NaN
315,nextar,NaN,NaN,NaN,False,NaN,NaN
341,pastry4,NaN,NaN,NaN,False,NaN,NaN


In [ ]:
# Search for food
ingredient = 'condiment'
food_query = ingredient
r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}&dataType=Foundation')
data = r.json()
data_df = pd.DataFrame.from_dict(data['foods'])
if len(data_df) == 0:  # Data not found
    r = requests.get(f'https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query={food_query}')
    data = r.json()
    data_df = pd.DataFrame.from_dict(data['foods'])
data_df

,fdcId,description,dataType,gtinUpc,publishedDate,brandOwner,brandName,ingredients,marketCountry,foodCategory,...,foodAttributeTypes,foodVersionIds,commonNames,additionalDescriptions,ndbNumber,shortDescription,subbrandName,preparationStateCode,footnote,gpcClassCode
0,2028508,CONDIMENT,Branded,756192001010,2021-10-28,Ham I Am! Inc.,HOGWASH,"BROWN SUGAR, HORSERADISH (HORSERADISH, DISTILL...",United States,"Ketchup, Mustard, BBQ & Cheese Sauce",...,[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,173320,"Fast foods, cheeseburger; double, large patty;...",SR Legacy,NaN,2019-04-01,NaN,NaN,NaN,NaN,Fast Foods,...,[],[],,,21396.0,NaN,NaN,NaN,NaN,NaN
2,173319,"Fast foods, cheeseburger; double, regular patt...",SR Legacy,NaN,2019-04-01,NaN,NaN,NaN,NaN,Fast Foods,...,[],[],,,21395.0,NaN,NaN,NaN,NaN,NaN
3,172082,"Fast foods, cheeseburger; single, large patty;...",SR Legacy,NaN,2019-04-01,NaN,NaN,NaN,NaN,Fast Foods,...,[],[],,,21398.0,NaN,NaN,NaN,NaN,NaN
4,170691,"Fast foods, cheeseburger; single, regular patt...",SR Legacy,NaN,2019-04-01,NaN,NaN,NaN,NaN,Fast Foods,...,[],[],,,21090.0,NaN,NaN,NaN,NaN,NaN
5,170710,"Fast foods, hamburger, large, single patty, wi...",SR Legacy,NaN,2019-04-01,NaN,NaN,NaN,NaN,Fast Foods,...,[],[],,,21202.0,NaN,NaN,NaN,NaN,NaN
6,170694,"Fast foods, hamburger; single, regular patty; ...",SR Legacy,NaN,2019-04-01,NaN,NaN,NaN,NaN,Fast Foods,...,[],[],,,21108.0,NaN,NaN,NaN,NaN,NaN
7,173323,"Fast foods, bagel, with breakfast steak, egg, ...",SR Legacy,NaN,2019-04-01,NaN,NaN,NaN,NaN,Fast Foods,...,[],[],,,21411.0,NaN,NaN,NaN,NaN,NaN
8,173322,"Fast foods, bagel, with egg, sausage patty, ch...",SR Legacy,NaN,2019-04-01,NaN,NaN,NaN,NaN,Fast Foods,...,[],[],,,21410.0,NaN,NaN,NaN,NaN,NaN
9,170293,"Fast foods, cheeseburger, double, regular patt...",SR Legacy,NaN,2019-04-01,NaN,NaN,NaN,NaN,Fast Foods,...,[],[],,,21094.0,NaN,NaN,NaN,NaN,NaN
